## 2. Procesamiento de datos simple

Una compañera de trabajo nos ha pedido un favor, aprovechando que sabe que estamos aprendiendo a programar. Tiene un histórico de partidos de fútbol catalán en un fichero, donde se almacenan los nombres de los equipos y los resultados. Necesita que procesemos los datos de forma automática, para extraer los resultados que necesita.

Utiliza el archivo "historic_partits.txt"

Necesita un programa que devuelva:

El número total de goles que ha marcado cada equipo.
El nombre del equipo más goleador.
El nombre del equipo más goleado
La clasificación global (cada victoria: 3 pts, empate 1 pts, derrota 0 pts)


In [ ]:
import re

def procesar_historico_partidos(nombre_archivo="Historico de Partidos.txt"):
    """
    Procesa un archivo de resultados de fútbol para calcular goles y una clasificación.
    """
    # 1. Inicializar el diccionario de estadísticas
    stats_equipos = {}
    
    # 2. Expresión regular para extraer datos del partido (Equipo1, Goles1, Goles2, Equipo2)
    # Patrón: "EquipoA\tGolesA-GolesB\tEquipoB" (se ajusta a tu formato de archivo)
    patron_partido = re.compile(r"([A-Za-z\s]+)\t(\d+)-(\d+)\t([A-Za-z\s]+)")

    try:
        with open(nombre_archivo, 'r', encoding='utf-8') as archivo:
            for linea in archivo:
                # Buscar y extraer los datos del partido
                match = patron_partido.search(linea.strip())
                if match:
                    # Desempaquetar los resultados
                    equipo_local, goles_local_str, goles_visitante_str, equipo_visitante = match.groups()
                    goles_local = int(goles_local_str)
                    goles_visitante = int(goles_visitante_str)
                    
                    # Inicializar los equipos si no están en las estadísticas
                    # Usamos un diccionario con valores por defecto (goles a favor, goles en contra, puntos)
                    for equipo in [equipo_local, equipo_visitante]:
                        if equipo not in stats_equipos:
                            # {equipo: [goles_favor, goles_contra, puntos]}
                            stats_equipos[equipo] = [0, 0, 0]

                    # 3. Actualizar Goles y Puntos
                    
                    # Puntos y Goles del Local
                    stats_equipos[equipo_local][0] += goles_local    # Goles a favor
                    stats_equipos[equipo_local][1] += goles_visitante # Goles en contra
                    
                    # Puntos y Goles del Visitante
                    stats_equipos[equipo_visitante][0] += goles_visitante # Goles a favor
                    stats_equipos[equipo_visitante][1] += goles_local # Goles en contra
                    
                    # Lógica de Puntuación
                    if goles_local > goles_visitante:        # Victoria Local
                        stats_equipos[equipo_local][2] += 3
                    elif goles_local < goles_visitante:      # Victoria Visitante
                        stats_equipos[equipo_visitante][2] += 3
                    else:                                    # Empate
                        stats_equipos[equipo_local][2] += 1
                        stats_equipos[equipo_visitante][2] += 1
                        
    except FileNotFoundError:
        print(f"Error: El archivo '{nombre_archivo}' no fue encontrado.")
        return None

    # 4. Extracción de Máximos y Clasificación Final
    
    equipo_mas_goleador = None
    max_goles_favor = -1
    equipo_mas_goleado = None
    max_goles_contra = -1
    
    clasificacion_final = [] # Almacenará tuplas (puntos, goles_favor, goles_contra, nombre_equipo)
    
    for nombre, (goles_favor, goles_contra, puntos) in stats_equipos.items():
        # Encontrar Máximo Goleador
        if goles_favor > max_goles_favor:
            max_goles_favor = goles_favor
            equipo_mas_goleador = nombre
            
        # Encontrar Máximo Goleado
        if goles_contra > max_goles_contra:
            max_goles_contra = goles_contra
            equipo_mas_goleado = nombre
            
        # Preparar para la clasificación (puntos, diferencia de gol, goles a favor)
        clasificacion_final.append((puntos, goles_favor - goles_contra, goles_favor, nombre))

    # Ordenar la clasificación: por Puntos (descendente), luego Dif. de Gol (descendente), luego Goles a Favor (descendente)
    # El orden en la tupla permite un ordenamiento simple con 'sorted'
    clasificacion_final.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)

    # 5. Mostrar Resultados
    
    print("=" * 60)
    print("REPORTE AUTOMÁTICO DE PARTIDOS")
    print("=" * 60)
    
    ## Reporte de Goles
    
    print("\n### Total de Goles Marcados por Equipo ###")
    
    # Ordenar por goles marcados para una mejor visualización
    goles_marcados_ordenados = sorted(
        [(stats[0], nombre) for nombre, stats in stats_equipos.items()],
        reverse=True
    )
    
    for goles, nombre in goles_marcados_ordenados:
        print(f"- **{nombre.ljust(20)}**: {goles} goles")
        
    print("-" * 60)
    print(f"**EQUIPO MÁS GOLEADOR**: {equipo_mas_goleador} (con {max_goles_favor} goles)")
    print(f"**EQUIPO MÁS GOLEADO**: {equipo_mas_goleado} (con {max_goles_contra} goles en contra)")
    print("-" * 60)
    
    ## Clasificación Final
    
    print("\n### 🏆 CLASIFICACIÓN GLOBAL (3 pts/Victoria, 1 pt/Empate) 🏆 ###")
    
    # Encabezados de la tabla
    header = "{:<4} {:<20} {:>5} {:>5} {:>5} {:>5}"
    print(header.format("Pos", "Equipo", "Pts", "GF", "GC", "DG"))
    print("-" * 60)

    for i, (puntos, dif_gol, goles_favor, nombre) in enumerate(clasificacion_final):
        goles_contra = stats_equipos[nombre][1] # Recuperar goles en contra para mostrar
        print(header.format(
            i + 1,
            nombre,
            puntos,
            goles_favor,
            goles_contra,
            dif_gol
        ))

# Ejecutar el programa
procesar_historico_partidos()